# Level 3B — Heavy-Tail and Jump Simulation

**Audience:** analysts who can run GBM scenarios and want models that allow
non-Gaussian log-return shapes.

**Prerequisites:** Level 3A and basic knowledge of distributions.

**Learning goals**

1. distinguish diffusion, compound-Poisson jump, random-clock, and stable-tail
   assumptions;
2. simulate Variance Gamma return and price paths;
3. simulate Merton jump-diffusion return and price paths;
4. simulate symmetric and skewed alpha-stable return and price paths;
5. interpret sample diagnostics without assuming nonexistent population
   moments.

**Outline:** model map → Variance Gamma → Merton jumps → skewed stable →
comparison → terminal wealth → exercise.

The Variance Gamma section follows Gamma-time-changed Brownian motion described
by Madan, Carr, and Chang (1998) and the owner-provided QC article. The jump
section follows Merton (1976). The stable section follows Mandelbrot's (1963)
stable Paretian proposal and uses the Chambers-Mallows-Stuck simulation
transformation.

## 1. Setup and model map

- **GBM:** continuous Gaussian log-return diffusion.
- **Variance Gamma:** Brownian motion evaluated on a random Gamma business
  clock; supports excess kurtosis and asymmetry.
- **Merton jump diffusion:** Gaussian diffusion plus a finite number of
  normally distributed log jumps arriving through a Poisson process.
- **Alpha-stable:** stable log increments with power-law tails; `beta` controls
  left/right tail asymmetry under the documented `S0` parameterization.

All functions return periodic **simple returns** so they can feed the same
terminal-wealth tools.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.analytics.risk import excess_kurtosis, skewness
from asset_management_toolkit.simulation import (
    simulate_gbm_returns,
    simulate_merton_jump_prices,
    simulate_merton_jump_returns,
    simulate_stable_prices,
    simulate_stable_returns,
    simulate_variance_gamma_prices,
    simulate_variance_gamma_returns,
    terminal_wealth_stats,
)

## 2. Variance Gamma: Brownian motion on a random clock

For \(\Delta t=1/P\), draw
\(G\sim\operatorname{Gamma}(\Delta t/\nu,\nu)\). Then:

\[
\Delta \log S =
(\ell-\theta)\Delta t+\theta G+\sigma\sqrt{G}Z
\]

The clock has \(E[G]=\Delta t\) and
\(\operatorname{Var}(G)=\nu\Delta t\). Parameter `theta` controls asymmetry,
while `variance_rate` (`ν`) controls how unevenly business time passes.

In [ ]:
vg_returns = simulate_variance_gamma_returns(
    n_years=3,
    n_scenarios=4_000,
    mean_log_return=0.07,
    theta=-0.10,
    volatility=0.15,
    variance_rate=0.20,
    periods_per_year=12,
    seed=42,
)
vg_returns.iloc[:5, :4]

In [ ]:
vg_prices = simulate_variance_gamma_prices(
    n_years=3,
    n_scenarios=4_000,
    mean_log_return=0.07,
    theta=-0.10,
    volatility=0.15,
    variance_rate=0.20,
    periods_per_year=12,
    initial_price=100.0,
    seed=42,
)
vg_prices.iloc[:, :20].plot(
    legend=False,
    title="Illustrative Variance Gamma price scenarios",
    xlabel="Monthly step",
    ylabel="Price index",
    figsize=(9, 4),
)

## 3. Merton jump diffusion: explicit event arrivals

For \(\Delta t=1/P\):

\[
\Delta \log S =
\left(\mu-\frac{1}{2}\sigma^2-\lambda\kappa\right)\Delta t
+\sigma\sqrt{\Delta t}Z
+\sum_{k=1}^{N_{\Delta t}}Y_k
\]

where \(N_{\Delta t}\sim\operatorname{Poisson}(\lambda\Delta t)\),
\(Y_k\sim\mathcal{N}(m_J,s_J^2)\), and
\(\kappa=\exp(m_J+s_J^2/2)-1\). The compensation term keeps expected price
growth governed by `expected_return`.

In [ ]:
merton_returns = simulate_merton_jump_returns(
    n_years=3,
    n_scenarios=4_000,
    expected_return=0.07,
    volatility=0.15,
    jump_intensity=1.5,
    jump_mean=-0.12,
    jump_volatility=0.20,
    periods_per_year=12,
    seed=42,
)
merton_prices = simulate_merton_jump_prices(
    n_years=3,
    n_scenarios=4_000,
    expected_return=0.07,
    volatility=0.15,
    jump_intensity=1.5,
    jump_mean=-0.12,
    jump_volatility=0.20,
    periods_per_year=12,
    initial_price=100.0,
    seed=42,
)
merton_prices.iloc[:, :20].plot(
    legend=False,
    title="Illustrative Merton jump-diffusion price scenarios",
    xlabel="Monthly step",
    ylabel="Price index",
    figsize=(9, 4),
)

## 4. Symmetric and skewed alpha-stable tails

For the symmetric case, stable increments obey the simple scaling rule:

\[
\Delta \log S =
\delta\Delta t+c\Delta t^{1/\alpha}X_{\alpha}
\]

Lower `alpha` produces heavier tails. At `alpha=2`, the stable law reaches its
Gaussian limit under the stable scale convention. For `alpha < 2`, population
variance is infinite; for `alpha <= 1`, the population mean is also undefined.

The general API uses Nolan's `S0` parameterization. `beta=0` is symmetric,
`beta<0` emphasizes the left tail, and `beta>0` emphasizes the right tail.
`beta` is a distribution parameter, not the ordinary third-moment skewness
coefficient. The simulator adjusts each step's location so the stated annual
`S0` parameters remain consistent under Lévy-process time scaling.

In [ ]:
stable_returns = simulate_stable_returns(
    n_years=3,
    n_scenarios=4_000,
    alpha=1.70,
    beta=0.0,
    scale=0.04,
    location=0.07,
    periods_per_year=12,
    seed=42,
)
left_skewed_returns = simulate_stable_returns(
    n_years=3,
    n_scenarios=4_000,
    alpha=1.70,
    beta=-0.60,
    scale=0.04,
    location=0.07,
    periods_per_year=12,
    seed=42,
)
left_skewed_prices = simulate_stable_prices(
    n_years=3,
    n_scenarios=4_000,
    alpha=1.70,
    beta=-0.60,
    scale=0.04,
    location=0.07,
    periods_per_year=12,
    initial_price=100.0,
    seed=42,
)
left_skewed_prices.iloc[:, :20].plot(
    legend=False,
    title="Illustrative left-skewed alpha-stable price scenarios",
    xlabel="Monthly step",
    ylabel="Price index",
    figsize=(9, 4),
)

## 5. Compare sample distributions

These are finite-sample diagnostics. They do not turn infinite stable
population moments into finite ones.

In [ ]:
gbm_returns = simulate_gbm_returns(
    n_years=3,
    n_scenarios=4_000,
    expected_return=0.07,
    volatility=0.15,
    periods_per_year=12,
    seed=42,
)

model_returns = {
    "GBM": gbm_returns.stack(),
    "Variance Gamma": vg_returns.stack(),
    "Merton jumps": merton_returns.stack(),
    "Stable beta=0.00": stable_returns.stack(),
    "Stable beta=-0.60": left_skewed_returns.stack(),
}
comparison = pd.DataFrame(
    {
        name: {
            "sample_mean": values.mean(),
            "sample_std": values.std(),
            "sample_skewness": skewness(values),
            "sample_excess_kurtosis": excess_kurtosis(values),
            "sample_min": values.min(),
            "sample_max": values.max(),
        }
        for name, values in model_returns.items()
    }
).T
comparison

## 6. Compare terminal outcomes

The same terminal-wealth function accepts every return matrix. Differences
come from model assumptions and parameters, not from a change in the wealth
calculation.

In [ ]:
terminal_comparison = pd.DataFrame(
    {
        name: terminal_wealth_stats(
            returns,
            initial_wealth=100.0,
            floor_wealth=80.0,
            cap_wealth=150.0,
        )
        for name, returns in {
            "GBM": gbm_returns,
            "Variance Gamma": vg_returns,
            "Merton jumps": merton_returns,
            "Stable beta=0.00": stable_returns,
            "Stable beta=-0.60": left_skewed_returns,
        }.items()
    }
)
terminal_comparison.loc[
    [
        "mean",
        "median",
        "standard_deviation",
        "probability_below_floor",
        "expected_shortfall_below_floor",
        "probability_above_cap",
    ]
]

## 7. Exercise — tail and skew sensitivity

1. Change stable `alpha` from 1.70 to 1.50 while holding `beta=-0.60`.
2. Change stable `beta` from -0.60 to +0.60.
3. Change Variance Gamma `theta` from -0.10 to +0.10.
4. Change Merton `jump_intensity` from 1.5 to 3.0.
5. Compare sample skewness, sample excess kurtosis, and floor-breach
   probability.

In [ ]:
# Build the alternative scenarios here.
stable_heavier = simulate_stable_returns(
    n_years=3,
    n_scenarios=4_000,
    alpha=1.50,
    beta=-0.60,
    scale=0.04,
    location=0.07,
    periods_per_year=12,
    seed=7,
)
stable_right_skewed = simulate_stable_returns(
    n_years=3,
    n_scenarios=4_000,
    alpha=1.70,
    beta=0.60,
    scale=0.04,
    location=0.07,
    periods_per_year=12,
    seed=7,
)
vg_positive_skew = simulate_variance_gamma_returns(
    n_years=3,
    n_scenarios=4_000,
    mean_log_return=0.07,
    theta=0.10,
    volatility=0.15,
    variance_rate=0.20,
    periods_per_year=12,
    seed=7,
)
merton_more_frequent = simulate_merton_jump_returns(
    n_years=3,
    n_scenarios=4_000,
    expected_return=0.07,
    volatility=0.15,
    jump_intensity=3.0,
    jump_mean=-0.12,
    jump_volatility=0.20,
    periods_per_year=12,
    seed=7,
)

### Answer scaffold

In [ ]:
exercise_returns = {
    "Stable alpha=1.50, beta=-0.60": stable_heavier,
    "Stable alpha=1.70, beta=+0.60": stable_right_skewed,
    "VG theta=+0.10": vg_positive_skew,
    "Merton intensity=3.0": merton_more_frequent,
}
pd.DataFrame(
    {
        name: {
            "sample_skewness": skewness(values.stack()),
            "sample_excess_kurtosis": excess_kurtosis(values.stack()),
            "probability_below_80": terminal_wealth_stats(
                values,
                initial_wealth=100.0,
                floor_wealth=80.0,
            )["probability_below_floor"],
        }
        for name, values in exercise_returns.items()
    }
).T

## Interpretation, pitfalls, and extensions

- These models generate scenarios; they do not identify which model will
  forecast future returns.
- Do not compare `scale` in a stable law directly with GBM volatility.
- State the stable parameterization. This toolkit uses Nolan `S0`; another
  library's `location` can differ when it uses `S1`.
- Do not report stable sample variance as a converged population variance when
  `alpha < 2`.
- Variance Gamma here is a statistical path simulator, not a calibrated
  risk-neutral option-pricing engine.
- Merton jumps here are physical/statistical scenarios. The parameters are not
  automatically risk-neutral option-pricing inputs.
- Report model, parameters, horizon, frequency, scenario count, and seed
  policy with every result.

Possible extensions include parameter calibration, CGMY, Kou double-exponential
jumps, stochastic volatility, and correlated multivariate scenarios. Each
needs a separate parameterization and validation contract.